# Particle Flow Network: quark/gluon classification

This notebook trains a **Particle Flow Network** on every constituent of each truth-matched jet.
A **jet** is a narrow spray of particles produced when a high-energy quark or gluon
turns into observable particles. Those particles are the jet's **constituents**.
The classification task is to use their measured patterns to estimate whether the
original particle was a quark or a gluon.

This is **supervised machine learning**: each training example has inputs (constituent
measurements) and a target answer (the simulation-derived quark/gluon label). It follows
the common preparation lesson and saves a self-describing model bundle.

A **Particle Flow Network (PFN)** applies the same small neural network to every particle,
turning it into a learned vector called an **embedding**. It sums all particle embeddings and
uses a second network to classify the whole jet. A sum is unchanged when its terms are
reordered, so this design builds permutation invariance into the architecture.

Set `QG_RUN_MODE=full` before launching Jupyter for the larger configuration. Quick
mode is the default. Both automatically use CUDA when it is available.


## 1. Environment and computing device

Run `./setup_student_env.sh` once before this lesson. A **Jupyter kernel** is the Python
process that actually executes notebook cells; selecting the henv kernel makes sure it
can see the packages installed for this course.

PyTorch can calculate on a CPU or on a GPU. A **GPU** performs many similar arithmetic
operations at once, which is useful for neural-network training. **CUDA** is the software
interface PyTorch uses to access an NVIDIA GPU. The code chooses CUDA automatically when
available, but gives the same lesson on a CPU.


In [ ]:
import importlib.util
required = ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'sklearn', 'torch', 'tqdm']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing packages: {missing}. From a terminal in this directory run "
        "./setup_student_env.sh (or use --current inside an existing henv), "
        "restart Jupyter from that henv, and select its registered kernel."
    )

import json, os, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
import qg_constituent_ml as qg

DEVICE = qg.choose_device()
RUN_MODE = os.getenv('QG_RUN_MODE', 'quick')
SOURCE = Path(os.getenv('QG_INPUT_PATH', 'data/inclusive_jets.parquet'))
print(f'PyTorch {torch.__version__}; built with CUDA {torch.version.cuda}')
print(f'device={DEVICE}' + (f'; GPU={torch.cuda.get_device_name(0)}' if DEVICE.type == 'cuda' else ''))
print(f'run mode={RUN_MODE}; source={SOURCE}')


## 2. Current data and model

Preparation is fingerprint-aware: changing the generated Parquet file creates a new
prepared-data namespace automatically. A **fingerprint** is a cryptographic summary
(SHA-256 here) of a file's exact bytes. Even a tiny file change produces a different
fingerprint, preventing us from silently pairing a model with the wrong dataset.

The printed **parameter count** is the number of adjustable numerical values in the neural
network. Training changes these values. More parameters can represent more complicated
patterns, but also require more data and can make overfitting easier.


In [ ]:
prepared = qg.prepare_dataset(SOURCE)
manifest = qg.load_manifest(prepared)
MODEL_CONFIG = qg.default_config('pfn', RUN_MODE)
model = qg.create_model('pfn', config=MODEL_CONFIG)
print(json.dumps({k: manifest[k] for k in ('source_sha256','n_jets','n_constituents','split_counts','class_counts')}, indent=2))
print(model)
print(f'parameters={sum(p.numel() for p in model.parameters()):,}; config={MODEL_CONFIG}')


### Canonical implementation

A neural network is built from mathematical **layers**. Each layer transforms numbers into
new numbers; during training, the network learns which transformations help predict the
label. The final raw output is a **logit**. Applying the sigmoid function converts it to a
score between 0 and 1, where larger values mean “more quark-like” in this project.

The reusable class lives in `qg_constituent_ml.py` so saved models can be reconstructed
later. Its source is displayed here to keep the architecture visible.


In [ ]:
import inspect
from IPython.display import Code, display
display(Code(inspect.getsource(qg.model_classes()['pfn']), language='python'))


## 3. Invariance checks

Jets contain different numbers of particles. To place several jets in one rectangular
tensor, shorter lists receive dummy entries called **padding**. A Boolean **mask** tells the
model which entries are real. Adding more dummy entries must not change a prediction.

The constituents form an unordered **set**, not a sentence: exchanging particle 2 and
particle 7 should not change the jet. This property is called **permutation invariance**.
The assertions below are unit tests for both physical requirements.


In [ ]:
loaders = qg.make_loaders(prepared, 'pfn', RUN_MODE)
batch = next(iter(loaders[2]))
model.eval()
with torch.no_grad():
    base = model(batch['features'][:4], batch['coords'][:4], batch['mask'][:4])
    order = torch.randperm(batch['features'].shape[1])
    permuted = model(batch['features'][:4, order], batch['coords'][:4, order], batch['mask'][:4, order])
    padded_f = torch.nn.functional.pad(batch['features'][:4], (0,0,0,3))
    padded_c = torch.nn.functional.pad(batch['coords'][:4], (0,0,0,3))
    padded_m = torch.nn.functional.pad(batch['mask'][:4], (0,3))
    padded = model(padded_f, padded_c, padded_m)
assert torch.allclose(base, permuted, atol=2e-5)
assert torch.allclose(base, padded, atol=2e-5)
print('Passed permutation and padding invariance checks.')


## 4. Train, validate, test, and save

During **training**, the model predicts labels, a **loss function** measures its errors,
and backpropagation computes how each parameter contributed to those errors. An optimizer
then makes a small parameter update. One pass through the training sample is an **epoch**.

The training batches are balanced so quarks and gluons contribute equally. A separate
**validation set** chooses when to stop and is never used for parameter updates. **Early
stopping** keeps the checkpoint from the epoch with the best validation result, limiting
overfitting. The **test set** is touched only for the final measurement and keeps the
naturally occurring class mixture.


In [ ]:
history, metrics, predictions = qg.train_model(model, loaders, 'pfn', RUN_MODE, DEVICE)
bundle = qg.save_model_bundle(model, 'pfn', RUN_MODE, MODEL_CONFIG, prepared,
                              history, metrics, predictions)
print(json.dumps(metrics, indent=2))
print(f'Saved model bundle: {bundle}')


In [ ]:
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].plot([x['epoch'] for x in history], [x['train_loss'] for x in history], marker='o')
axes[0].set(xlabel='epoch', ylabel='BCE loss', title='Training history')
fpr,tpr,_=roc_curve(predictions['labels'], predictions['scores'])
axes[1].plot(tpr,1/np.clip(fpr,1e-3,None),label=f"AUC={metrics['roc_auc']:.3f}")
axes[1].set(xlabel='quark efficiency',ylabel='gluon rejection',yscale='log',title='Held-out performance')
axes[1].legend(); plt.tight_layout(); plt.show()


## 5. Reading the plots and result

**Binary cross-entropy (BCE)** is the training loss: smaller values mean that the predicted
scores agree better with the known labels. The **ROC curve** scans every possible score
threshold. Quark efficiency is the fraction of true quark jets retained; gluon rejection
is the inverse of the fraction of gluon jets mistakenly retained. **AUC** summarizes the
ROC curve: 0.5 is random ordering and 1.0 is perfect ordering on this sample.

Compare architectures only through the evaluation notebook, which enforces matching dataset
and split fingerprints. A larger network on a small sample is not automatically a better
physics model: it may learn statistical fluctuations (**overfit**) instead of patterns that
generalize to unseen jets.
